# Entrega 2 - Álgebra Linear

## Redução de dimensionalidade dos indicadores do agente

Na Entrega 1, cada estudante foi representado por um vetor com seis características acadêmicas, financeiras e de relacionamento. Nesta entrega, aplicamos **Análise de Componentes Principais (PCA)** para reduzir essa matriz de seis dimensões para duas componentes.

A PCA utiliza **autovalores e autovetores** da matriz de covariância. Os autovetores definem novas direções para representar os dados e os autovalores indicam quanta informação existe em cada direção.

O objetivo é verificar se os perfis dos estudantes podem ser resumidos em menos dimensões, facilitando a visualização e a identificação dos fatores que mais diferenciam os casos.

## Como executar

1. Mantenha este notebook ao lado de uma pasta chamada `Base`.
2. Coloque na pasta `Base`: `Contato.xlsx`, `Matriculas.xlsx`, `Financeiro.xlsx`, `Historico.xlsx` e `Relacionamentos.xlsx`.
3. Instale as dependências com `pip install pandas numpy openpyxl jupyter`.
4. Inicie o Jupyter Notebook ou JupyterLab.
5. Abra este arquivo, selecione um kernel Python 3 e execute **Run All Cells**.

As saídas desta versão já estão salvas. As bases não devem ser publicadas em repositórios abertos.

## 1. Preparação dos dados

O tratamento segue a Entrega 1: selecionamos estudantes ativos em 2026-2, usamos o histórico acadêmico até 2026-1 e reunimos informações acadêmicas, financeiras e de relacionamento pelo `ID_ALUNO`.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

nomes_arquivos = [
    "Contato.xlsx", "Matriculas.xlsx", "Financeiro.xlsx",
    "Historico.xlsx", "Relacionamentos.xlsx"
]

def possui_bases(pasta):
    return pasta.exists() and all((pasta / nome).exists() for nome in nomes_arquivos)

candidatos = [
    Path("Base"), Path("."), Path("../Base"), Path(".."),
    Path.home() / "Downloads" / "Base",
    Path.home() / "Downloads" / "Entrega_Algebra_Linear" / "Base",
]
downloads = Path.home() / "Downloads"
if downloads.exists():
    candidatos.extend(downloads.glob("*/Base"))

BASE_DIR = next((p.resolve() for p in candidatos if possui_bases(p)), None)
if BASE_DIR is None:
    raise FileNotFoundError(
        "Crie uma pasta Base ao lado do notebook e coloque nela os cinco arquivos Excel."
    )

arquivos = {nome.replace(".xlsx", "").lower(): BASE_DIR / nome for nome in nomes_arquivos}
print(f"Bases localizadas em: {BASE_DIR}")

Bases localizadas em: C:\Users\sribe\Downloads\Entrega_Algebra_Linear\Base


In [2]:
# Estudantes ativos em 2026-2.
col_status = "Status Acadêmico (Período Letivo Atual) (Período Letivo - Guideme)"
matriculas = pd.read_excel(
    arquivos["matriculas"],
    usecols=["ID_ALUNO", "Período Letivo Atual", col_status],
).drop_duplicates()

status_ativos = {"Matriculado", "Pré-Matriculado", "Aguardando Rematrícula"}
coorte = (
    matriculas.loc[
        (matriculas["Período Letivo Atual"].astype(str) == "2026-2")
        & matriculas[col_status].isin(status_ativos),
        ["ID_ALUNO"],
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f"Estudantes ativos: {len(coorte):,}")

Estudantes ativos: 4,000


In [3]:
# Indicadores financeiros.
financeiro = pd.read_excel(
    arquivos["financeiro"],
    usecols=[
        "ID_ALUNO", "SERVICOPARCELA", "LANCAMENTOVALORORIGINAL",
        "STATUSLANCAMENTO", "LANCAMENTODATAVENCIMENTO", "LANCAMENTODATABAIXA",
    ],
)
financeiro = financeiro.dropna(
    subset=[
        "SERVICOPARCELA", "LANCAMENTOVALORORIGINAL", "STATUSLANCAMENTO",
        "LANCAMENTODATAVENCIMENTO", "LANCAMENTODATABAIXA",
    ],
    how="all",
).drop_duplicates()
financeiro = financeiro[financeiro["ID_ALUNO"].isin(coorte["ID_ALUNO"])].copy()

financeiro["LANCAMENTODATAVENCIMENTO"] = pd.to_datetime(
    financeiro["LANCAMENTODATAVENCIMENTO"], errors="coerce"
)
financeiro["LANCAMENTODATABAIXA"] = pd.to_datetime(
    financeiro["LANCAMENTODATABAIXA"], errors="coerce"
)
data_referencia = pd.Timestamp("2026-08-31")
data_inicial = pd.Timestamp("2020-01-01")
vencimento_valido = financeiro["LANCAMENTODATAVENCIMENTO"].between(data_inicial, data_referencia)
baixa_valida = financeiro["LANCAMENTODATABAIXA"].between(data_inicial, data_referencia)

financeiro["PENDENCIA_VENCIDA"] = (
    financeiro["STATUSLANCAMENTO"].eq("Em Aberto")
    & vencimento_valido
).astype(int)

status_pago = financeiro["STATUSLANCAMENTO"].isin(
    ["Baixado", "Baixado por Acordo", "Baixado parcialmente"]
)
financeiro["DIAS_ATRASO"] = (
    financeiro["LANCAMENTODATABAIXA"] - financeiro["LANCAMENTODATAVENCIMENTO"]
).dt.days
financeiro.loc[~(status_pago & vencimento_valido & baixa_valida), "DIAS_ATRASO"] = np.nan
financeiro["DIAS_ATRASO"] = financeiro["DIAS_ATRASO"].clip(lower=0, upper=365)

fin_por_aluno = financeiro.groupby("ID_ALUNO", as_index=False).agg(
    Pendencias_Financeiras=("PENDENCIA_VENCIDA", "sum"),
    Dias_Atraso_Medio=("DIAS_ATRASO", "mean"),
)

In [4]:
# Indicadores acadêmicos.
historico = pd.read_excel(
    arquivos["historico"],
    usecols=[
        "ID_ALUNO", "Ano/Semestre Letivo", "Situação", "Nota Obtida",
        "Nota Distribuída", "Faltas Lançadas", "Aulas Ministradas",
    ],
).drop_duplicates()
historico = historico[
    historico["ID_ALUNO"].isin(coorte["ID_ALUNO"])
    & (historico["Ano/Semestre Letivo"].astype(str) <= "2026-1")
].copy()

for coluna in ["Nota Obtida", "Nota Distribuída", "Faltas Lançadas", "Aulas Ministradas"]:
    historico[coluna] = pd.to_numeric(historico[coluna], errors="coerce")

concluido = historico["Situação"].isin({"Aprovado", "Reprovado", "Não Concluído"})
nota_valida = concluido & (historico["Nota Distribuída"] > 0)
faltas_validas = concluido & (historico["Aulas Ministradas"] > 0)

historico["NOTA_PADRONIZADA"] = np.nan
historico.loc[nota_valida, "NOTA_PADRONIZADA"] = (
    10 * historico.loc[nota_valida, "Nota Obtida"]
    / historico.loc[nota_valida, "Nota Distribuída"]
).clip(0, 10)

historico["PERCENTUAL_FALTAS"] = np.nan
historico.loc[faltas_validas, "PERCENTUAL_FALTAS"] = (
    100 * historico.loc[faltas_validas, "Faltas Lançadas"]
    / historico.loc[faltas_validas, "Aulas Ministradas"]
).clip(0, 100)
historico["REPROVADO"] = historico["Situação"].eq("Reprovado").astype(int)

hist_por_aluno = historico.groupby("ID_ALUNO", as_index=False).agg(
    Media_Notas=("NOTA_PADRONIZADA", "mean"),
    Percentual_Faltas=("PERCENTUAL_FALTAS", "mean"),
    Reprovacoes=("REPROVADO", "sum"),
)

In [5]:
# Indicador de relacionamento.
relacionamentos = pd.read_excel(
    arquivos["relacionamentos"],
    usecols=["ID_ALUNO", "Código", "Tema Relacionamento", "Data de Criação"],
).drop_duplicates(subset=["Código"])
relacionamentos["Data de Criação"] = pd.to_datetime(
    relacionamentos["Data de Criação"], errors="coerce"
)
relacionamentos = relacionamentos[
    relacionamentos["ID_ALUNO"].isin(coorte["ID_ALUNO"])
    & (relacionamentos["Data de Criação"] <= data_referencia)
].copy()
relacionamentos["REGISTRO_RISCO"] = relacionamentos["Tema Relacionamento"].eq(
    "Risco de Evasão"
).astype(int)

rel_por_aluno = relacionamentos.groupby("ID_ALUNO", as_index=False).agg(
    Registros_Risco=("REGISTRO_RISCO", "sum")
)

## 2. Matriz da Entrega 1

Cada linha representa um estudante e cada coluna representa um sinal de acompanhamento:

\[
X = [r_{nota}, r_{faltas}, r_{reprovações}, r_{pendências}, r_{atraso}, r_{relacionamento}]
\]

As características ficam entre 0 e 1. Valores maiores indicam sinais mais intensos.

In [6]:
base = (
    coorte.merge(hist_por_aluno, on="ID_ALUNO", how="left")
    .merge(fin_por_aluno, on="ID_ALUNO", how="left")
    .merge(rel_por_aluno, on="ID_ALUNO", how="left")
)
base[["Reprovacoes", "Pendencias_Financeiras", "Dias_Atraso_Medio", "Registros_Risco"]] = (
    base[["Reprovacoes", "Pendencias_Financeiras", "Dias_Atraso_Medio", "Registros_Risco"]]
    .fillna(0)
)
base = base.dropna(subset=["Media_Notas", "Percentual_Faltas"]).copy()

def normalizar_positivo(serie):
    serie = pd.to_numeric(serie, errors="coerce").fillna(0).clip(lower=0)
    limite = serie.quantile(0.95)
    return serie.clip(upper=limite) / limite if limite > 0 else pd.Series(0.0, index=serie.index)

base["Risco_Nota"] = (1 - base["Media_Notas"] / 10).clip(0, 1)
base["Risco_Faltas"] = (base["Percentual_Faltas"] / 100).clip(0, 1)
base["Risco_Reprovacoes"] = normalizar_positivo(base["Reprovacoes"])
base["Risco_Pendencias"] = normalizar_positivo(base["Pendencias_Financeiras"])
base["Risco_Atraso"] = normalizar_positivo(base["Dias_Atraso_Medio"])
base["Risco_Relacionamento"] = normalizar_positivo(base["Registros_Risco"])

colunas = [
    "Risco_Nota", "Risco_Faltas", "Risco_Reprovacoes",
    "Risco_Pendencias", "Risco_Atraso", "Risco_Relacionamento",
]
rotulos = ["Nota", "Faltas", "Reprovações", "Pendências", "Atraso", "Relacionamento"]
X = base[colunas].to_numpy(dtype=float)
print(f"Matriz da Entrega 1: {X.shape[0]} estudantes x {X.shape[1]} características")

Matriz da Entrega 1: 3383 estudantes x 6 características


## 3. Padronização e matriz de covariância

Antes da PCA, centralizamos e padronizamos as colunas:

\[
Z_{ij} = \frac{X_{ij} - \bar{X}_j}{s_j}
\]

Em seguida, calculamos a matriz de covariância:

\[
C = \frac{Z^T Z}{n-1}
\]

Essa matriz mostra como as seis características variam em conjunto.

In [7]:
medias = X.mean(axis=0)
desvios = X.std(axis=0, ddof=1)
desvios[desvios == 0] = 1
Z = (X - medias) / desvios

C = (Z.T @ Z) / (len(Z) - 1)
matriz_covariancia = pd.DataFrame(C, index=rotulos, columns=rotulos)
matriz_covariancia

,Nota,Faltas,Reprovações,Pendências,Atraso,Relacionamento
Nota,1.0000,0.3668,0.6725,0.1150,0.1445,0.1670
Faltas,0.3668,1.0000,0.4226,0.2812,0.1770,0.1255
Reprovações,0.6725,0.4226,1.0000,0.2618,0.2213,0.1762
Pendências,0.1150,0.2812,0.2618,1.0000,0.3470,0.1089
Atraso,0.1445,0.1770,0.2213,0.3470,1.0000,0.0516
Relacionamento,0.1670,0.1255,0.1762,0.1089,0.0516,1.0000


## 4. Autovalores e autovetores

Resolvemos a equação:

\[
C v = \lambda v
\]

Cada autovetor representa uma nova direção da matriz. O autovalor correspondente mede a quantidade de variância explicada por essa direção.

In [8]:
autovalores, autovetores = np.linalg.eigh(C)
ordem = np.argsort(autovalores)[::-1]
autovalores = autovalores[ordem]
autovetores = autovetores[:, ordem]

# O sinal de um autovetor é arbitrário. Ajustamos apenas para facilitar a leitura.
for j in range(autovetores.shape[1]):
    maior = np.argmax(np.abs(autovetores[:, j]))
    if autovetores[maior, j] < 0:
        autovetores[:, j] *= -1

variancia_explicada = autovalores / autovalores.sum()
resultado_pca = pd.DataFrame({
    "Componente": [f"CP{i}" for i in range(1, 7)],
    "Autovalor": autovalores,
    "Variância explicada (%)": 100 * variancia_explicada,
    "Variância acumulada (%)": 100 * np.cumsum(variancia_explicada),
})
resultado_pca

,Componente,Autovalor,Variância explicada (%),Variância acumulada (%)
0,CP1,2.3178,38.6293,38.6293
1,CP2,1.1233,18.7211,57.3504
2,CP3,0.9320,15.5325,72.8830
3,CP4,0.7238,12.0635,84.9465
4,CP5,0.5930,9.8834,94.8299
5,CP6,0.3102,5.1701,100.0000


## 5. Redução para duas dimensões

Selecionamos os dois autovetores associados aos maiores autovalores. A transformação linear é:

\[
Y = ZW
\]

`W` contém os dois autovetores escolhidos e `Y` é a nova matriz com duas colunas: CP1 e CP2.

In [9]:
W = autovetores[:, :2]
Y = Z @ W
base["CP1"] = Y[:, 0]
base["CP2"] = Y[:, 1]

cargas = pd.DataFrame(W, index=rotulos, columns=["CP1", "CP2"])
print(f"Variância preservada por CP1 e CP2: {100 * variancia_explicada[:2].sum():.2f}%")
cargas

Variância preservada por CP1 e CP2: 57.35%


,CP1,CP2
Nota,0.4936,-0.4160
Faltas,0.4474,-0.0312
Reprovações,0.5444,-0.2510
Pendências,0.3451,0.5939
Atraso,0.3051,0.6172
Relacionamento,0.2184,-0.1710


### Interpretação das componentes

- **CP1 - intensidade acadêmica:** os maiores pesos aparecem em reprovações (0,5444), nota (0,4936) e faltas (0,4474). Valores altos nessa componente representam uma combinação mais intensa de sinais acadêmicos.
- **CP2 - dimensão financeira:** os maiores pesos aparecem em atraso (0,6172) e pendências financeiras (0,5939). Essa componente ajuda a separar casos predominantemente financeiros dos casos predominantemente acadêmicos.

As duas componentes preservam **57,35% da variância total**. Portanto, elas oferecem um resumo útil para visualização e triagem, mas ainda deixam 42,65% da variação nas outras quatro componentes. Por esse motivo, os seis indicadores originais continuam necessários para explicar cada caso.

## 6. Análise dos perfis em duas dimensões

Após a transformação, cada estudante possui apenas duas coordenadas. Dividimos o plano em quatro quadrantes e verificamos quais sinais originais predominam em cada região.

In [10]:
dimensao_principal = base[colunas].idxmax(axis=1).map(dict(zip(colunas, rotulos)))
base["Dimensão principal"] = dimensao_principal
base["Quadrante"] = np.select(
    [
        (base["CP1"] >= 0) & (base["CP2"] >= 0),
        (base["CP1"] < 0) & (base["CP2"] >= 0),
        (base["CP1"] < 0) & (base["CP2"] < 0),
    ],
    ["CP1+ / CP2+", "CP1- / CP2+", "CP1- / CP2-"],
    default="CP1+ / CP2-",
)

distribuicao = pd.crosstab(base["Quadrante"], base["Dimensão principal"])
distribuicao

Dimensão principal,Atraso,Faltas,Nota,Pendências,Relacionamento,Reprovações
Quadrante,,,,,,
CP1+ / CP2+,119,0,9,416,43,55
CP1+ / CP2-,1,0,186,49,186,272
CP1- / CP2+,47,4,143,634,17,1
CP1- / CP2-,0,0,1079,31,83,8


In [11]:
# Casos mais afastados do centro da projeção, úteis para revisão prioritária.
base["Distância no plano"] = np.linalg.norm(Y, axis=1)
casos_extremos = base.nlargest(10, "Distância no plano")[[
    "ID_ALUNO", "CP1", "CP2", "Distância no plano", "Dimensão principal"
]].reset_index(drop=True)
casos_extremos

,ID_ALUNO,CP1,CP2,Distância no plano,Dimensão principal
0,7573,5.9436,-3.6377,6.9685,Relacionamento
1,17681,5.9849,-3.3589,6.8630,Reprovações
2,12758,5.5398,-3.7922,6.7134,Reprovações
3,17101,6.4551,0.5374,6.4775,Reprovações
4,4896,5.5081,-3.2234,6.3819,Reprovações
5,15577,5.8740,-2.4831,6.3773,Relacionamento
6,10229,5.2946,-3.5376,6.3677,Relacionamento
7,3480,6.1853,1.3248,6.3256,Reprovações
8,7676,5.4443,-3.0915,6.2609,Nota
9,10988,6.0618,-1.4329,6.2288,Nota


## 7. Perda de informação

Também reconstruímos aproximadamente a matriz original usando apenas CP1 e CP2. O erro quadrático médio mostra a diferença entre os valores originais e os reconstruídos.

In [12]:
Z_reconstruida = Y @ W.T
X_reconstruida = Z_reconstruida * desvios + medias
erro_medio = np.mean((X - X_reconstruida) ** 2)

print(f"Erro quadrático médio da reconstrução: {erro_medio:.6f}")
print(f"Redução realizada: {X.shape[1]} dimensões para {Y.shape[1]} dimensões")

Erro quadrático médio da reconstrução: 0.029633
Redução realizada: 6 dimensões para 2 dimensões


## 8. Aplicação no projeto

A PCA melhora a análise do agente de três formas:

- **Monitoramento:** permite representar milhares de estudantes em um plano de duas dimensões e localizar perfis extremos ou grupos próximos.
- **Pendências:** as cargas dos autovetores mostram quais características acadêmicas, financeiras ou de relacionamento mais influenciam cada componente.
- **Orientação ao estudante:** estudantes com posições próximas no gráfico possuem combinações semelhantes de sinais e podem receber checklists iniciais parecidos, sempre com validação humana.

A redução não substitui os seis indicadores originais. Ela funciona como uma visão resumida para análise e priorização.

## 9. Conclusão

A matriz da Entrega 1 foi transformada por meio dos autovetores da matriz de covariância. A PCA criou duas novas componentes que resumem os principais padrões presentes nos dados.

Com isso, tornou-se possível representar 3.383 estudantes em duas dimensões, identificar quais fatores explicam as diferenças entre os perfis e medir a informação preservada após a redução. A CP1 resumiu principalmente sinais acadêmicos e a CP2 destacou sinais financeiros. O erro quadrático médio da reconstrução foi 0,029633.

O agente passa a ter uma visão mais simples dos casos, mantendo os indicadores originais disponíveis para explicar cada situação e apoiar a validação humana.